In [2]:
pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 5.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd

INPUT_FULL = "messages.csv"
OUT_CSV    = "purchased_only.csv"
CHUNKSIZE  = 1_000_000  # RAM에 맞춰 조절 (500k~2M)

# 기존 출력 파일 제거 (깨끗하게 시작)
if os.path.exists(OUT_CSV):
    os.remove(OUT_CSV)

first = True
read_csv_kwargs = {"chunksize": CHUNKSIZE}

# 핵심: 청크 모드에서는 pyarrow 엔진 사용 금지 (미지원)
# 대신 dtype_backend="pyarrow"는 사용할 수 있지만, 청크마다 판다스 변환이 들어가므로
# 여기선 호환성 우선으로 생략.
try:
    import pyarrow  # 설치 여부만 확인 (엔진은 사용 X)
    # 참고: 전체 파일을 한 번에 읽을 때만 아래처럼 사용 가능
    # read_csv_kwargs["engine"] = "pyarrow"  # <-- 청크 모드에서는 주석
except Exception:
    pass

def is_positive(s: pd.Series) -> pd.Series:
    """1 / '1' / 't' / 'T' / 'true' / 'True' / True 를 모두 True로 간주"""
    # 빠른 경로: 이미 불리언이면 그대로
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)

    # 숫자형(0/1) 처리
    out = pd.Series(False, index=s.index)
    # 결측은 우선 0 취급
    s2 = s.fillna(0)

    # 숫자/불리언 1 또는 True
    try:
        out |= (pd.to_numeric(s2, errors="coerce") == 1)
    except Exception:
        pass

    # 문자열 변형 후 비교
    s_str = s2.astype(str).str.strip().str.lower()
    out |= s_str.isin({"1", "t", "true", "y", "yes"})

    return out.fillna(False)

for chunk in pd.read_csv(INPUT_FULL, **read_csv_kwargs):
    if "is_purchased" not in chunk.columns:
        raise ValueError("is_purchased 컬럼이 없습니다.")

    mask = is_positive(chunk["is_purchased"])
    pos = chunk.loc[mask]
    if not pos.empty:
        pos.to_csv(OUT_CSV, mode="w" if first else "a", index=False, header=first)
        first = False

print({"csv": OUT_CSV})


/tmp/ipykernel_1293/574695411.py:48: DtypeWarning: Columns (8,19,21,23,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_FULL, **read_csv_kwargs):
/tmp/ipykernel_1293/574695411.py:48: DtypeWarning: Columns (8,19,21,23,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_FULL, **read_csv_kwargs):
/tmp/ipykernel_1293/574695411.py:48: DtypeWarning: Columns (8,19,21,23,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_FULL, **read_csv_kwargs):
/tmp/ipykernel_1293/574695411.py:48: DtypeWarning: Columns (21,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(INPUT_FULL, **read_csv_kwargs):
/tmp/ipykernel_1293/574695411.py:48: DtypeWarning: Columns (25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch

{'csv': 'purchased_only.csv'}


In [3]:
import os
import pandas as pd
import numpy as np

# 파일 경로 및 설정
INPUT_DEMO = "messages-demo.csv"      # 초기 데모 데이터
INPUT_EXTRACTED = "purchased_only.csv"  # 이전에 추출한 positive 데이터
OUTPUT_FINAL = "messages-final.csv" # 최종 주입 데이터
TARGET_TOTAL_POS = 43352            # 최종 목표 positive 수
SEED = 42                           # 재현성 위한 시드
UNIQUE_KEY = "id"                   # 고유 키 (데이터셋에 맞게 수정 필요)

# 기존 출력 파일 제거
if os.path.exists(OUTPUT_FINAL):
    os.remove(OUTPUT_FINAL)

# 랜덤 시드 고정
np.random.seed(SEED)
rng = np.random.RandomState(SEED)

# 초기 messages-demo.csv 로드 및 positive 식별
demo_df = pd.read_csv(INPUT_DEMO)
if "is_purchased" not in demo_df.columns:
    raise ValueError("is_purchased 컬럼이 없습니다.")

# demo_df의 positive (is_purchased = 1) 식별
demo_df["is_purchased"] = demo_df["is_purchased"].apply(lambda x: 1 if str(x).strip().lower() in {"1", "t", "true", "y", "yes"} else 0)
original_positives = demo_df[demo_df["is_purchased"] == 1]
original_pos_count = len(original_positives)
print(f"Original positives in messages-demo.csv: {original_pos_count}")

# demo_df의 positive 고유 키 저장
if UNIQUE_KEY not in demo_df.columns:
    raise ValueError(f"{UNIQUE_KEY} 컬럼이 없습니다. 고유 식별자 컬럼을 지정하세요.")
demo_pos_keys = set(original_positives[UNIQUE_KEY].values)

# purchased_only.csv 로드 (추출된 positive)
extracted_df = pd.read_csv(INPUT_EXTRACTED)
if "is_purchased" not in extracted_df.columns:
    raise ValueError("is_purchased 컬럼이 없습니다.")
extracted_df["is_purchased"] = extracted_df["is_purchased"].apply(lambda x: 1 if str(x).strip().lower() in {"1", "t", "true", "y", "yes"} else 0)
extracted_positives = extracted_df[extracted_df["is_purchased"] == 1]

# 중복 제거 (고유 키 기준)
extracted_positives = extracted_positives.drop_duplicates(subset=[UNIQUE_KEY])

# demo와 겹치지 않는 샘플 필터링
non_overlapping = extracted_positives[~extracted_positives[UNIQUE_KEY].isin(demo_pos_keys)]
needed_add = TARGET_TOTAL_POS - original_pos_count

if len(non_overlapping) >= needed_add:
    # 필요한 수만큼 랜덤 샘플링
    additional_pos = non_overlapping.sample(n=needed_add, random_state=SEED)
else:
    raise ValueError(f"Non-overlapping positives after deduplication ({len(non_overlapping)})가 목표 {needed_add}보다 적습니다. 더 많은 데이터 추출 필요.")

# messages-demo.csv에 추가 positive 주입
final_df = pd.concat([demo_df, additional_pos], ignore_index=True)

# 중복 제거 (최종 데이터셋에서도 안전성 위해)
final_df = final_df.drop_duplicates(subset=[UNIQUE_KEY])

# 최종 positive 수 확인
final_positives = final_df["is_purchased"].sum()
print(f"Final positives in messages-final.csv: {final_positives}")

## 코드 추가 ##
# is_purchased 컬럼을 다시 원래의 'f'/'t' 형태로 변환
final_df['is_purchased'] = final_df['is_purchased'].map({0: 'f', 1: 't'})
print("\n'is_purchased' column converted back to 'f'/'t' format.")

# CSV 저장
final_df.to_csv(OUTPUT_FINAL, index=False)
print({"output_csv": OUTPUT_FINAL, "final_positives": final_positives})

/tmp/ipykernel_44564/1281912986.py:22: DtypeWarning: Columns (7,8,16,17,19,21,23,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  demo_df = pd.read_csv(INPUT_DEMO)


Original positives in messages-demo.csv: 12340


/tmp/ipykernel_44564/1281912986.py:38: DtypeWarning: Columns (21,23,25) have mixed types. Specify dtype option on import or set low_memory=False.
  extracted_df = pd.read_csv(INPUT_EXTRACTED)


Final positives in messages-final.csv: 43352

'is_purchased' column converted back to 'f'/'t' format.
{'output_csv': 'messages-final.csv', 'final_positives': np.int64(43352)}


In [4]:
import pandas as pd
a=pd.read_csv('messages-final.csv')
a

/tmp/ipykernel_29934/1551678221.py:2: DtypeWarning: Columns (7,8,16,17,19,21,23,25,27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  a=pd.read_csv('messages-final.csv')


,id,message_id,campaign_id,message_type,client_id,channel,category,platform,email_provider,stream,...,is_soft_bounced,soft_bounced_at,is_complained,complained_at,is_blocked,blocked_at,is_purchased,purchased_at,created_at,updated_at
0,3527358,3f6aaad3-bab7-4886-b083-fe8c1f210066,31,transactional,1515915625489833514,email,NaN,NaN,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,t,2021-05-06 16:40:38,2023-04-27 08:55:05.883908,2023-04-27 08:57:33.080129
1,3527619,0e670ecc-4549-44f6-86ed-469682d34837,32,transactional,1515915625489220445,email,NaN,NaN,yandex.ru,desktop,...,f,NaN,f,NaN,f,NaN,f,NaN,2023-04-27 08:55:06.265821,2023-04-27 08:56:18.60223
2,3527980,276b25cf-1bda-4faf-b3a4-98e4161f9357,32,transactional,1515915625489854185,email,NaN,NaN,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,f,NaN,2023-04-27 08:55:06.777039,2023-04-27 08:56:19.112546
3,3528369,4545aff2-09b3-45e3-9abd-c680357e5429,32,transactional,1515915625489101550,email,NaN,NaN,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,f,NaN,2023-04-27 08:55:07.325906,2023-04-27 08:56:19.590637
4,3528648,5850858d-2dcf-4f31-a0d3-5db5649b17c4,32,transactional,1515915625490455948,email,NaN,NaN,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,f,NaN,2023-04-27 08:55:07.727792,2023-04-27 08:56:19.926474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10031007,48082192,1515915625487410834-18012-em-64070425,18012,trigger,1515915625487410834,email,NaN,smartphone,yandex.ru,desktop,...,f,NaN,f,NaN,f,NaN,t,2023-03-07 09:45:31,2023-04-28 23:08:25.803667,2023-04-28 23:08:25.806711
10031008,87019357,1515915625468208625-1508-6165324791691,1508,bulk,1515915625468208625,email,NaN,desktop,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,t,2021-10-12 08:28:31,2023-04-29 20:50:43.122589,2023-04-29 20:50:43.127083
10031009,470878313,1515915625764195556-7879-63748780a93b7,7879,bulk,1515915625764195556,email,NaN,smartphone,yandex.ru,desktop,...,f,NaN,f,NaN,f,NaN,t,2022-11-18 14:30:49,2023-05-10 01:19:35.215474,2023-05-10 03:36:20.372497
10031010,112863351,1515915625487929623-2088-61a9c91c6b5f7,2088,bulk,1515915625487929623,email,NaN,desktop,mail.ru,desktop,...,f,NaN,f,NaN,f,NaN,t,2021-12-03 08:10:11,2023-04-30 11:31:29.432518,2023-04-30 11:31:29.43964


In [6]:
a['platform'].value_counts(dropna=False)

platform
NaN           9272056
desktop        564227
smartphone     182302
phablet          8017
tablet           4338
android            72
Name: count, dtype: int64